# Setup: your knowledge graph environment

**Time**: ~15 minutes. **Cost**: Effectively $0 to set up — the graph database has a permanently free tier, and the test calls in this notebook use a handful of tokens on the cheap OpenRouter model from Week 2.

> **How to run this notebook.** Open it in Cursor, VS Code, or Jupyter on your own computer. Create this week's environment once (`cd` into the week folder, then `uv venv` and `uv pip install -r requirements.txt`), select that `.venv` as the kernel, and keep your OpenRouter key in the project `.env` file. Full walkthrough: [Week 2's setup guide](../week_2/lesson08_setup_guide.ipynb).

This week adds one more moving part to the pipeline you've been building since Week 2. Up to now, every notebook has talked to one thing: an AI model, over an API. This week you're also talking to a **graph database** — a separate system that stores nodes (entities) and the typed relationships between them, and answers questions by traversing those relationships rather than by scanning rows.

That means two accounts instead of one, and two connections to test before you touch any real content. This notebook gets both working: a free graph database instance, and the same kind of swappable LLM connection you set up in Week 2, generalized so the rest of this week's notebook can call whichever model provider you configure with a single line.

## A note on privacy

Same rule as every other week: work only with public example data, synthetic data, or the anonymized examples this course provides. The documents you'll build a graph from this week are entirely invented — a fictional company, fictional customers, fictional tickets — so there's nothing to protect, but the habit matters more than this specific exercise. A knowledge graph built from real internal documents is, by construction, a concentrated store of exactly the kind of information Week 4 covered: names, complaints, internal decisions. Treat a real one accordingly.

Data-use policy for prompts you send through OpenRouter depends on the model behind the slug and on OpenRouter's terms — check those rather than assuming. The graph database provider (Neo4j) is a separate company with its own terms; the free tier used here is a hosted instance on their infrastructure, so don't put anything in it you wouldn't put in any other third-party cloud service.

## Part 1: get a graph database

### What is Neo4j, and why Aura instead of installing something?

**Neo4j** is a graph database — software built specifically to store nodes and relationships and to answer traversal queries efficiently, the way a spreadsheet is built specifically for rows and formulas. **Cypher** is Neo4j's query language, the graph equivalent of SQL; Week 7's overview linked its [getting-started guide](https://neo4j.com/docs/getting-started/cypher/), and you'll write real Cypher later in this notebook and the next one.

Neo4j can run two ways: installed locally (Neo4j Desktop, a program on your own machine), or hosted for you (**Aura**, Neo4j's managed cloud service). This course uses Aura so you don't have to install Neo4j Desktop: **AuraDB Free** is a permanently free, hosted instance — no credit card, no time limit, created with a Google or GitHub login in about a minute ([Neo4j's own FAQ and pricing pages](https://neo4j.com/docs/aura/)). If you already run Neo4j Desktop locally and would rather point this notebook at that instead, the connection cell below works the same way against a local address (`bolt://localhost:7687`) — the only thing that changes is which URI you put in `.env`. The rest of this guide assumes Aura.

**What the free tier actually gives you**, and where documentation disagrees: Neo4j's own pages are not fully consistent with each other on the exact ceiling — one source states 50,000 nodes and 175,000 relationships, another states 200,000 nodes and 400,000 relationships. Either number is far more than this week's exercise needs (our graph will have well under 100 nodes), so the discrepancy doesn't matter for coursework, but don't quote a specific number to a stakeholder without checking the current limit in your own Aura console first — this is exactly the "verify before stating as fact" habit the course has been building. One number both sources agree on: a Free instance **auto-pauses after 72 hours of inactivity** (your data is retained, it just needs waking up — open the console and it resumes in under a minute) and is **deleted if left paused for 90 days**. If you come back to this notebook after a break and the connection cell fails, check the Aura console first before assuming something is broken.

### Steps to create your instance

1. Go to [console.neo4j.io](https://console.neo4j.io) and sign in (Google, GitHub, or email — no card required).
2. Click **New instance**, choose **AuraDB Free**.
3. Give it a name (e.g. `ai4tm-session5`) and a region close to you.
4. Click **Create**. Neo4j will show you a **connection URI, username, and a generated password exactly once** — download the credentials file or copy all three now. If you lose the password, you'll need to reset it from the console; Neo4j does not store it for you to look up later.
5. Wait for the instance status to show **Running** (usually under a minute).

## Part 2: store your graph database credentials

Week 2's API key: a `.env` file in the project folder, not pasted into a cell. Add three more lines:

```
NEO4J_URI=neo4j+s://xxxxxxxx.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=the-generated-password
```

`NEO4J_URI` starts with `neo4j+s://`. The username is usually `neo4j`. Reload the notebook kernel after you save `.env` so the new values are picked up.

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

for name in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    value = os.getenv(name)
    if value:
        shown = value[:12] + "..." if name == "NEO4J_URI" else "*" * 8
        print(f"{name}: loaded ({shown})")
    else:
        print(f"{name}: NOT FOUND -- add it to your .env file.")

## Part 3: install the graph database driver and connect

`neo4j` is Neo4j's official Python driver — the library that turns Python function calls into Cypher queries sent over the network to your Aura instance, the same role the OpenAI library plays for OpenRouter.

In [ ]:
import pathlib

_cwd = pathlib.Path.cwd()
_week = "week_6"
if _cwd.name == _week and (_cwd / "requirements.txt").exists():
    _req_path = "requirements.txt"
elif (_cwd / _week / "requirements.txt").exists():
    _req_path = f"{_week}/requirements.txt"
elif (_cwd.parent / _week / "requirements.txt").exists():
    _req_path = f"../{_week}/requirements.txt"
else:
    _req_path = None

if _req_path is None:
    print("Couldn't find", _week + "/requirements.txt", "from:", _cwd)
    print("Set up the local environment first -- see Week 2's setup guide.")
else:
    %pip install -q -r {_req_path}

print("Driver installed.")

In [ ]:
from neo4j import GraphDatabase
import os

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD")),
)

# A minimal Cypher query, just to prove the connection works.
# RETURN behaves like SELECT in SQL: it hands back a value without touching any stored data.
with driver.session() as session:
    result = session.run("RETURN 1 AS ok")
    record = result.single()
    print("Connection ok. Neo4j returned:", record["ok"])

If that errored, check in order: the instance status is **Running** in the Aura console (not paused or still provisioning), the URI starts with `neo4j+s://` with no typos, and the three names in `.env` match exactly (`NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`). Restart the kernel after editing `.env`.

## Part 4: one OpenAI client, any OpenRouter model

Week 2 set up a single OpenRouter connection through the OpenAI Python library. This week's task keeps that pattern: one client, one helper function, `call_llm(prompt)`. To switch models, change the `MODEL_NAME` line below. You do not need a second library to try Claude, Gemini, or another OpenRouter slug — the client stays the same.

In [ ]:
print("Using the OpenAI library already installed from week_6/requirements.txt.")

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import OpenAI

# Walk up from the notebook folder to find the project `.env`.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

_api_key = os.getenv("OPENROUTER_API_KEY")
if not _api_key:
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Add it to a .env file in the project folder "
        "(see week_2/lesson08_setup_guide.ipynb)."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=_api_key,
)

MODEL_NAME = "qwen/qwen3-8b"  # change this line to switch models
print("Connected. Model:", MODEL_NAME)


def call_llm(prompt: str, model: str = None, temperature: float = 0) -> str:
    """Send `prompt` to the OpenRouter model and return the text response.

    Every notebook this week calls this function instead of the client
    directly, so MODEL_NAME is the only thing that needs to change to switch models.
    """
    response = client.chat.completions.create(
        model=model or MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content

Model slugs change over time. If a call errors mentioning the model name specifically, check [openrouter.ai/models](https://openrouter.ai/models) rather than assuming this notebook is wrong. Swap models by changing the `MODEL_NAME` line in this notebook.

In [ ]:
# Test the connection through call_llm(), not through a provider client directly --
# this is the function the rest of this week's notebook will use.
test_answer = call_llm("In one sentence, what is a knowledge graph?")
print(test_answer)

## You're set up

Both connections work: a graph database to hold the knowledge graph, and a swappable LLM call for extraction and querying. The next notebook, [`knowledge_graph_pipeline.ipynb`](knowledge_graph_pipeline.ipynb), builds a graph from a set of internal documents and queries it — including asking a model to answer a question using only what the graph actually contains, the technique called GraphRAG.

**Troubleshooting reference:**

| Symptom | Likely cause |
|---|---|
| `NEO4J_URI not found` | Name typo in `.env`, or the kernel was not restarted after saving |
| Connection hangs or times out | Instance is paused (open the Aura console to resume it) or still provisioning |
| `Unauthorized` from Neo4j | Password copied incorrectly at creation — reset it from the console |
| `call_llm` errors mentioning a model name | That provider renamed or retired the model; check their current model list |
| Everything worked yesterday, fails today with no code changes | Aura Free auto-pauses after 72 hours idle — check the console status first |